# Repeating E5-small queries on BEIR SciFact: analysis

This notebook reads immutable inference and evaluation artifacts. It does not run stages or pipelines. Add the exact completed run IDs below, then execute from the `query-repetition` project root.


In [ ]:
from pathlib import Path

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

from retrieval_core.utils.analysis import (
    build_analysis_frames,
    load_metrics_frame,
    metric_comparison_table,
    plot_metric_comparison,
    plot_query_metric_comparison,
)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Run this notebook from the query-repetition project root.")

INFERENCE_RUNS: dict[str, str] = {
    # "baseline": "query-repetition-e5-small-scifact--baseline",
    # "repeated": "query-repetition-e5-small-scifact--repeated",
}
EVALUATION_RUNS: dict[str, str] = {
    # "baseline": "query-repetition-e5-small-scifact--baseline-evaluation",
    # "repeated": "query-repetition-e5-small-scifact--repeated-evaluation",
}
BASELINE = "baseline"
TREATMENT = "repeated"
TARGET_METRICS = ["NDCG@10", "Recall@10", "MRR@50", "Recall@50", "Recall@100", "HitRate@10"]

## Experiment and hypothesis

This experiment compares the unchanged `intfloat/e5-small-v2` query pipeline with a treatment that repeats each raw query twice before standard E5 preprocessing. The preregistered hypothesis is that repetition produces a higher `NDCG@10` on the BEIR SciFact test split.


## Runs

These are the exact immutable runs used throughout the report.


In [ ]:
run_rows = [
    {"stage": stage, "label": label, "run_id": run_id}
    for stage, runs in [("inference", INFERENCE_RUNS), ("evaluation", EVALUATION_RUNS)]
    for label, run_id in runs.items()
]
display(pd.DataFrame(run_rows, columns=["stage", "label", "run_id"]))

## Target metrics

`NDCG@10` is primary. Deltas are treatment minus baseline.


In [ ]:
if EVALUATION_RUNS:
    metrics_df = load_metrics_frame(EVALUATION_RUNS, project_root=PROJECT_ROOT)
    comparison_df = metric_comparison_table(metrics_df, baseline=BASELINE, treatment=TREATMENT)
    comparison_df = comparison_df.loc[comparison_df["metric"].isin(TARGET_METRICS)]
    display(comparison_df.style.format({BASELINE: "{:.6f}", TREATMENT: "{:.6f}", "delta": "{:+.6f}"}))
    plot_metric_comparison(comparison_df, baseline=BASELINE, treatment=TREATMENT)
else:
    metrics_df = pd.DataFrame()
    comparison_df = pd.DataFrame()
    print("Add exact evaluation run IDs to render aggregate results.")

## Query-level diagnostics

Predictions are resolved through each inference manifest and joined to the qrels recorded in the first run's resolved configuration. Chunk predictions are collapsed to source documents to match evaluation semantics.


In [ ]:
if INFERENCE_RUNS:
    predictions_df, query_summary_df, qrels_df = build_analysis_frames(
        INFERENCE_RUNS, project_root=PROJECT_ROOT
    )
    display(query_summary_df.groupby("run_label")[["reciprocal_rank", "recall_at_run_depth"]].mean())
    plot_query_metric_comparison(
        query_summary_df, baseline=BASELINE, treatment=TREATMENT, metric="reciprocal_rank"
    )
    paired = query_summary_df.pivot(index="query_id", columns="run_label", values="reciprocal_rank")
    paired["delta"] = paired[TREATMENT] - paired[BASELINE]
    display(pd.Series({"improved": paired["delta"].gt(0).sum(), "degraded": paired["delta"].lt(0).sum(), "unchanged": paired["delta"].eq(0).sum()}, name="query_count").to_frame())
else:
    predictions_df = pd.DataFrame()
    query_summary_df = pd.DataFrame()
    qrels_df = pd.DataFrame()
    print("Add exact inference run IDs to render query-level diagnostics.")

## Conclusion

Summarize whether the observed `NDCG@10` delta supports the preregistered hypothesis. Keep claims limited to SciFact and distinguish the metric result from possible explanations.
